# Download Dataset

In [ ]:
!pip install catboost

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For time series
from typing import List
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

In [ ]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [ ]:
download_kaggle("kaggle competitions download -c individual-test-spai-sorting-hat")


--- Download complete! ---


# Explore Dataset

In [ ]:
df_attendance = pd.read_csv("/content/attendance.csv")
df_attendance

,user_id,datetime
0,5931fa6a-af6b-43b8-babb-28349854d406,2024-04-19 17:30:08.827555
1,5f1a2ecf-943d-446b-b94d-d7d1ad64213d,2024-04-19 17:30:11.405020
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,2024-04-19 17:30:14.433098
3,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453
4,f922a5d4-3651-46f7-aa98-307c045dd4b5,2024-04-19 17:30:21.556805
...,...,...
10798,ef836255-9ff1-48bd-9ee1-b3e205f96367,2024-05-17 13:57:11.458189
10799,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449
10800,1e103917-1d29-424d-b5e1-924e0cdcd6dc,2024-05-17 13:57:19.924906
10801,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543


In [ ]:
len(df_attendance["user_id"].unique())

170

In [ ]:
df_attendance.isna().sum().sum()

np.int64(0)

In [ ]:
df_house = pd.read_csv("/content/train.csv")
df_house

,user_id,house
0,4b2569be-cd32-40aa-9cc4-9e13e55903de,house1
1,cdd098ee-4295-4839-b1a3-4d44840feacd,house3
2,226c662f-b1b3-40bc-9bfc-4f49a80b9041,house6
3,7889a31f-7351-4b4d-b872-e04e5e35b9dd,house2
4,0492584f-5dda-4ad6-ab2f-79c8d43a8753,house1
...,...,...
97,7b3f7524-f201-44df-a929-5b92eac9457b,house5
98,6f16696b-22a0-4f3c-b2ad-f6b9e0256775,house3
99,272f295e-c673-4c17-8172-b9342b9d1192,house6
100,1f7d7fe6-a09f-45c9-87f1-5bec07a066fa,house2


In [ ]:
len(df_house["user_id"].unique())

102

In [ ]:
df_submission = pd.read_csv("/content/submission.csv")
df_submission

,user_id,house
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,NaN
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,NaN
...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,NaN
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,NaN
65,1bedb648-5a50-4680-a09b-27240759f9a0,NaN
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,NaN


In [ ]:
len(df_submission["user_id"].unique())

68

## Explore Dataset

In [ ]:
def extract_time_features(df: pd.DataFrame, time_columns: list = None) -> pd.DataFrame:
    """
    Extracts time-based features from specified columns in a DataFrame.
    If time_columns is not provided, it attempts to infer time columns
    by checking common time-related column names.

    Args:
        df (pd.DataFrame): The input DataFrame.
        time_columns (list, optional): A list of column names containing time data.
                                        If None, common time-related column names will be checked.

    Returns:
        pd.DataFrame: The DataFrame with new time feature columns.
    """

    df_copy = df.copy()

    # Common time-related column name patterns for inference
    if time_columns is None:
        possible_time_col_patterns = [
            'time', 'date', 'timestamp', 'datetime', 'created_at',
            'updated_at', 'ts', 'event_time', 'start_time', 'end_time'
        ]

        inferred_time_columns = [
            col for col in df_copy.columns
            if any(pattern in col.lower() for pattern in possible_time_col_patterns)
        ]

        if not inferred_time_columns:
            print("Warning: No time columns specified and no common time-related columns found. "
                  "Returning original DataFrame.")
            return df_copy
        time_columns = inferred_time_columns
        print(f"Inferred time columns: {time_columns}")

    print(f"{time_columns} is valid")


    # Possible time formats to try for robust parsing
    possible_time_formats = [
        "%Y-%m-%d %H:%M:%S.%f",  # 2025-07-05 13:42:29.123456
        "%Y-%m-%d %H:%M:%S",    # 2025-07-05 13:42:29
        "%Y-%m-%dT%H:%M:%S.%fZ",# ISO 8601 with Z (UTC)
        "%Y-%m-%dT%H:%M:%S",    # ISO 8601 without Z
        "%Y/%m/%d %H:%M:%S",    # 2025/07/05 13:42:29
        "%d-%m-%Y %H:%M:%S",    # 05-07-2025 13:42:29
        "%m/%d/%Y %I:%M:%S %p", # 07/05/2025 01:42:29 PM
        "%d/%m/%Y %H:%M",       # 05/07/2025 13:42
        "%Y-%m-%d",             # 2025-07-05
        "%d-%m-%Y",             # 05-07-2025
        "%m/%d/%Y",             # 07/05/2025
        "%H:%M:%S",             # 13:42:29
        "%H:%M",                # 13:42
        "%I:%M %p"              # 01:42 PM
    ]

    for col in time_columns:
        if col not in df_copy.columns:
            print(f"Warning: Column '{col}' not found in DataFrame. Skipping.")
            continue

        print("Column is okay")
        print(col)

        # Convert to datetime, coercing errors to NaT (Not a Time)
        # We try multiple formats with errors='coerce' to handle mixed formats
        df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)
        print(df_copy)

        # Drop rows where conversion failed for this column, or handle as NaT
        # For simplicity in feature extraction, we'll proceed with NaT values
        # If a column is entirely NaT after conversion, it's probably not a time column
        if df_copy[f'{col}_dt'].isnull().all() and not df_copy[col].isnull().all():
            print(f"Warning: Column '{col}' could not be converted to datetime. Skipping feature extraction for this column.")
            df_copy = df_copy.drop(columns=[f'{col}_dt'])
            continue

        # Extract features
        dt_col = df_copy[f'{col}_dt'].dt
        df_copy[f'{col}_year'] = dt_col.year
        df_copy[f'{col}_month'] = dt_col.month
        df_copy[f'{col}_day'] = dt_col.day
        df_copy[f'{col}_hour'] = dt_col.hour
        df_copy[f'{col}_minute'] = dt_col.minute
        df_copy[f'{col}_second'] = dt_col.second
        df_copy[f'{col}_dayofweek'] = dt_col.dayofweek # Monday=0, Sunday=6
        df_copy[f'{col}_dayofyear'] = dt_col.dayofyear
        df_copy[f'{col}_quarter'] = dt_col.quarter
        df_copy[f'{col}_is_weekend'] = dt_col.dayofweek.isin([5, 6]).astype(int)
        df_copy[f'{col}_season'] = (dt_col.month % 12 + 3) // 3 # Simple season mapping
        df_copy[f'{col}_is_month_start'] = dt_col.is_month_start.astype(int)
        df_copy[f'{col}_is_month_end'] = dt_col.is_month_end.astype(int)
        df_copy[f'{col}_is_quarter_start'] = dt_col.is_quarter_start.astype(int)
        df_copy[f'{col}_is_quarter_end'] = dt_col.is_quarter_end.astype(int)

        # Optional: Add cyclical features (sin/cos transformations for hour, dayofweek, etc.)
        df_copy[f'{col}_hour_sin'] = np.sin(2 * np.pi * dt_col.hour / 24)
        df_copy[f'{col}_hour_cos'] = np.cos(2 * np.pi * dt_col.hour / 24)

    return df_copy

In [ ]:
df_attendance = extract_time_features(df=df_attendance, time_columns=["datetime"])
df_attendance

['datetime'] is valid
Column is okay
datetime
                                    user_id                    datetime  \
0      5931fa6a-af6b-43b8-babb-28349854d406  2024-04-19 17:30:08.827555   
1      5f1a2ecf-943d-446b-b94d-d7d1ad64213d  2024-04-19 17:30:11.405020   
2      6e009a96-2b8b-4048-8b0c-20a7924a4b4f  2024-04-19 17:30:14.433098   
3      cdd098ee-4295-4839-b1a3-4d44840feacd  2024-04-19 17:30:17.214453   
4      f922a5d4-3651-46f7-aa98-307c045dd4b5  2024-04-19 17:30:21.556805   
...                                     ...                         ...   
10798  ef836255-9ff1-48bd-9ee1-b3e205f96367  2024-05-17 13:57:11.458189   
10799  84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f  2024-05-17 13:57:15.457449   
10800  1e103917-1d29-424d-b5e1-924e0cdcd6dc  2024-05-17 13:57:19.924906   
10801  9207b26d-79dc-41fc-b62c-67a6821132b9  2024-05-17 14:01:03.323543   
10802  25e69851-885b-4cd1-8c32-e29e604714ea  2024-05-17 14:01:05.950781   

                     datetime_dt  
0     2024-04-19 1

/tmp/ipython-input-195-2477569942.py:68: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_copy[f'{col}_dt'] = pd.to_datetime(df_copy[col], errors='coerce', infer_datetime_format=True)


,user_id,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos
0,5931fa6a-af6b-43b8-babb-28349854d406,2024-04-19 17:30:08.827555,2024-04-19 17:30:08.827555,2024.0,4.0,19.0,17.0,30.0,8.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
1,5f1a2ecf-943d-446b-b94d-d7d1ad64213d,2024-04-19 17:30:11.405020,2024-04-19 17:30:11.405020,2024.0,4.0,19.0,17.0,30.0,11.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,2024-04-19 17:30:14.433098,2024-04-19 17:30:14.433098,2024.0,4.0,19.0,17.0,30.0,14.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
3,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,2024-04-19 17:30:17.214453,2024.0,4.0,19.0,17.0,30.0,17.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
4,f922a5d4-3651-46f7-aa98-307c045dd4b5,2024-04-19 17:30:21.556805,2024-04-19 17:30:21.556805,2024.0,4.0,19.0,17.0,30.0,21.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10798,ef836255-9ff1-48bd-9ee1-b3e205f96367,2024-05-17 13:57:11.458189,2024-05-17 13:57:11.458189,2024.0,5.0,17.0,13.0,57.0,11.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10799,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,2024-05-17 13:57:15.457449,2024.0,5.0,17.0,13.0,57.0,15.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10800,1e103917-1d29-424d-b5e1-924e0cdcd6dc,2024-05-17 13:57:19.924906,2024-05-17 13:57:19.924906,2024.0,5.0,17.0,13.0,57.0,19.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10801,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,2024-05-17 14:01:03.323543,2024.0,5.0,17.0,14.0,1.0,3.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.500000,-0.866025


In [ ]:
df_attendance_house = pd.merge(df_attendance, df_house, how="inner", on=["user_id"])
df_attendance_house = df_attendance_house.dropna()
df_attendance_house

,user_id,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,...,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,2024-04-19 17:30:17.214453,2024.0,4.0,19.0,17.0,30.0,17.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house3
1,2ad18754-2522-4fa9-af12-bccdf42d3905,2024-04-19 17:30:24.008379,2024-04-19 17:30:24.008379,2024.0,4.0,19.0,17.0,30.0,24.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
2,3bd6dcac-904c-4614-be50-9c3949cb3405,2024-04-19 17:30:26.610108,2024-04-19 17:30:26.610108,2024.0,4.0,19.0,17.0,30.0,26.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,2024-04-19 17:30:29.393161,2024-04-19 17:30:29.393161,2024.0,4.0,19.0,17.0,30.0,29.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house5
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,2024-04-19 17:30:43.212998,2024-04-19 17:30:43.212998,2024.0,4.0,19.0,17.0,30.0,43.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819,house3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,2024-05-17 13:57:03.521711,2024-05-17 13:57:03.521711,2024.0,5.0,17.0,13.0,57.0,3.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house4
6480,3179b05a-487a-4208-9d7a-115c9532149b,2024-05-17 13:57:06.084968,2024-05-17 13:57:06.084968,2024.0,5.0,17.0,13.0,57.0,6.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house4
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,2024-05-17 13:57:15.457449,2024.0,5.0,17.0,13.0,57.0,15.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926,house3
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,2024-05-17 14:01:03.323543,2024.0,5.0,17.0,14.0,1.0,3.0,4.0,...,2.0,0,2.0,0,0,0,0,-0.500000,-0.866025,house3


In [ ]:
df_attendance_house.isna().sum().sum()

np.int64(0)

In [ ]:
df_attendance_house.describe()

,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos
count,6392,6392.0,6392.000000,6392.000000,6392.000000,6392.000000,6392.000000,6392.000000,6392.000000,6392.0,6392.000000,6392.0,6392.000000,6392.000000,6392.0,6392.0,6.392000e+03,6392.000000
mean,2024-05-03 08:50:19.138029056,2024.0,4.588705,15.162234,12.374061,41.722622,29.233573,2.191646,123.823373,2.0,0.118429,2.0,0.045056,0.044431,0.0,0.0,-4.609335e-02,-0.643079
min,2024-04-18 23:09:37.495417,2024.0,4.000000,1.000000,8.000000,0.000000,0.000000,0.000000,109.000000,2.0,0.000000,2.0,0.000000,0.000000,0.0,0.0,-1.000000e+00,-1.000000
25%,2024-04-24 17:36:54.569717248,2024.0,4.000000,8.000000,8.000000,35.000000,14.000000,1.000000,115.000000,2.0,0.000000,2.0,0.000000,0.000000,0.0,0.0,-9.659258e-01,-1.000000
50%,2024-05-02 17:55:42.327009024,2024.0,5.000000,15.000000,12.000000,46.000000,29.000000,2.000000,123.000000,2.0,0.000000,2.0,0.000000,0.000000,0.0,0.0,1.224647e-16,-0.500000
75%,2024-05-10 14:22:10.562774272,2024.0,5.000000,22.000000,17.000000,51.000000,44.000000,3.000000,131.000000,2.0,0.000000,2.0,0.000000,0.000000,0.0,0.0,8.660254e-01,-0.258819
max,2024-05-17 14:01:05.950781,2024.0,5.000000,30.000000,23.000000,59.000000,59.000000,6.000000,138.000000,2.0,1.000000,2.0,1.000000,1.000000,0.0,0.0,8.660254e-01,0.965926
std,NaN,0.0,0.492107,8.438061,3.423588,12.672291,17.198521,1.708824,8.706965,0.0,0.323141,0.0,0.207444,0.206066,0.0,0.0,6.964989e-01,0.315129


In [ ]:
df_attendance_house = df_attendance_house.drop(
    columns=[
        "datetime", "datetime_dt", "datetime_month",
        "datetime_year", "datetime_quarter", "datetime_season",
        "datetime_is_month_start", "datetime_is_month_end", "datetime_is_quarter_start",
        "datetime_is_quarter_end"
    ]
)

df_attendance_house

,user_id,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_is_weekend,datetime_hour_sin,datetime_hour_cos,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,19.0,17.0,30.0,17.0,4.0,110.0,0,-0.965926,-0.258819,house3
1,2ad18754-2522-4fa9-af12-bccdf42d3905,19.0,17.0,30.0,24.0,4.0,110.0,0,-0.965926,-0.258819,house5
2,3bd6dcac-904c-4614-be50-9c3949cb3405,19.0,17.0,30.0,26.0,4.0,110.0,0,-0.965926,-0.258819,house5
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,19.0,17.0,30.0,29.0,4.0,110.0,0,-0.965926,-0.258819,house5
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,19.0,17.0,30.0,43.0,4.0,110.0,0,-0.965926,-0.258819,house3
...,...,...,...,...,...,...,...,...,...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,17.0,13.0,57.0,3.0,4.0,138.0,0,-0.258819,-0.965926,house4
6480,3179b05a-487a-4208-9d7a-115c9532149b,17.0,13.0,57.0,6.0,4.0,138.0,0,-0.258819,-0.965926,house4
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,17.0,13.0,57.0,15.0,4.0,138.0,0,-0.258819,-0.965926,house3
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,17.0,14.0,1.0,3.0,4.0,138.0,0,-0.500000,-0.866025,house3


## Create Features For Late Time Attendance

In [ ]:
def create_attendance_bool(df_attendance_house):

  df_attendance_house["Late_Morning"] = (
      (df_attendance_house["datetime_hour"] >= 9) &
      (df_attendance_house["datetime_minute"] >= 0) &
        (df_attendance_house["datetime_hour"] <= 11) &
      (df_attendance_house["datetime_is_weekend"] == 0)).astype(int)



  df_attendance_house["Late_Evening"] = (
      (df_attendance_house["datetime_hour"] >= 13) &
      (df_attendance_house["datetime_minute"] >= 0) &
        (df_attendance_house["datetime_hour"] <= 15) &
              # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)



  df_attendance_house["Late_Night"] = (
      # Normal Day Attendance
      (df_attendance_house["datetime_hour"] >= 18) &
      (df_attendance_house["datetime_minute"] >= 0) &
      (df_attendance_house["datetime_hour"] <= 19) &

      # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  df_attendance_house["Late_Morning_in_5"] = (
      (df_attendance_house["datetime_hour"] >= 9) &
      (df_attendance_house["datetime_minute"] >= 0) &
        (df_attendance_house["datetime_hour"] <= 11) &
      (df_attendance_house["datetime_is_weekend"] == 0)).astype(int)


  df_attendance_house["Late_Evening_in_5"] = (
      (df_attendance_house["datetime_hour"] >= 13) &
      (df_attendance_house["datetime_minute"] >= 0) &
        (df_attendance_house["datetime_hour"] <= 15) &
              # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  df_attendance_house["Late_Nighe_in_5"] = (
      # Normal Day Attendance
      (df_attendance_house["datetime_hour"] >= 18) &
      (df_attendance_house["datetime_minute"] >= 5) &
      (df_attendance_house["datetime_hour"] <= 19) &

      # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  df_attendance_house["Early_Morning"] = (
      (df_attendance_house["datetime_hour"] >= 8) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 9) &
          (df_attendance_house["datetime_minute"] <= 45) &
      (df_attendance_house["datetime_is_weekend"] == 0)).astype(int)


  df_attendance_house["Early_Evening"] = (
      (df_attendance_house["datetime_hour"] >= 12) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 13) &
          (df_attendance_house["datetime_minute"] <= 45) &
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  df_attendance_house["Early_Night"] = (
      (df_attendance_house["datetime_hour"] >= 17) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 18) &
          (df_attendance_house["datetime_minute"] <= 45) &
      # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)



  df_attendance_house["In_time_Morning"] = (
      (df_attendance_house["datetime_hour"] >= 8) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 9) &
      (df_attendance_house["datetime_is_weekend"] == 0)).astype(int)


  df_attendance_house["In_time_Evening"] = (
      (df_attendance_house["datetime_hour"] >= 12) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 13) &
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  df_attendance_house["In_time_Night"] = (
      (df_attendance_house["datetime_hour"] >= 17) &
      (df_attendance_house["datetime_minute"] >= 30) &
        (df_attendance_house["datetime_hour"] <= 18) &
      # We dont check the name in weekend
      (df_attendance_house["datetime_is_weekend"] == 0)
  ).astype(int)


  return df_attendance_house

In [ ]:
df_attendance_house = create_attendance_bool(df_attendance_house)

In [ ]:
df_attendance_house["datetime_dayofweek"].unique()

array([4., 5., 6., 0., 3., 1., 2.])

In [ ]:
len(df_attendance_house[ df_attendance_house["Late_Night"] == 1])

7

## Mapping

In [ ]:
df_attendance_house["house"].unique()

array(['house3', 'house5', 'house2', 'house4', 'house1', 'house6'],
      dtype=object)

In [ ]:
house_mapping = {
  'house1': 0,
  'house2': 1,
  'house3': 2,
  'house4': 3,
  'house5': 4,
  'house6': 5,
}

df_attendance_house['house'] = df_attendance_house['house'].map(house_mapping)
df_attendance_house

,user_id,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_is_weekend,datetime_hour_sin,datetime_hour_cos,...,Late_Night,Late_Morning_in_5,Late_Evening_in_5,Late_Nighe_in_5,Early_Morning,Early_Evening,Early_Night,In_time_Morning,In_time_Evening,In_time_Night
0,cdd098ee-4295-4839-b1a3-4d44840feacd,19.0,17.0,30.0,17.0,4.0,110.0,0,-0.965926,-0.258819,...,0,0,0,0,0,0,1,0,0,1
1,2ad18754-2522-4fa9-af12-bccdf42d3905,19.0,17.0,30.0,24.0,4.0,110.0,0,-0.965926,-0.258819,...,0,0,0,0,0,0,1,0,0,1
2,3bd6dcac-904c-4614-be50-9c3949cb3405,19.0,17.0,30.0,26.0,4.0,110.0,0,-0.965926,-0.258819,...,0,0,0,0,0,0,1,0,0,1
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,19.0,17.0,30.0,29.0,4.0,110.0,0,-0.965926,-0.258819,...,0,0,0,0,0,0,1,0,0,1
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,19.0,17.0,30.0,43.0,4.0,110.0,0,-0.965926,-0.258819,...,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,17.0,13.0,57.0,3.0,4.0,138.0,0,-0.258819,-0.965926,...,0,0,1,0,0,0,0,0,1,0
6480,3179b05a-487a-4208-9d7a-115c9532149b,17.0,13.0,57.0,6.0,4.0,138.0,0,-0.258819,-0.965926,...,0,0,1,0,0,0,0,0,1,0
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,17.0,13.0,57.0,15.0,4.0,138.0,0,-0.258819,-0.965926,...,0,0,1,0,0,0,0,0,1,0
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,17.0,14.0,1.0,3.0,4.0,138.0,0,-0.500000,-0.866025,...,0,0,1,0,0,0,0,0,0,0


In [ ]:
house_table = df_attendance_house[["user_id", "house"]]

In [ ]:
df_attendance_house["house"].unique()

array([2, 4, 1, 3, 0, 5])

In [ ]:
df_attendance_house.columns

Index(['user_id', 'datetime_day', 'datetime_hour', 'datetime_minute',
       'datetime_second', 'datetime_dayofweek', 'datetime_dayofyear',
       'datetime_is_weekend', 'datetime_hour_sin', 'datetime_hour_cos',
       'house', 'Late_Morning', 'Late_Evening', 'Late_Night',
       'Late_Morning_in_5', 'Late_Evening_in_5', 'Late_Nighe_in_5',
       'Early_Morning', 'Early_Evening', 'Early_Night', 'In_time_Morning',
       'In_time_Evening', 'In_time_Night'],
      dtype='object')

In [ ]:
house_table = df_attendance_house[["user_id", "house"]]
house_table

,user_id,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,2
1,2ad18754-2522-4fa9-af12-bccdf42d3905,4
2,3bd6dcac-904c-4614-be50-9c3949cb3405,4
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,4
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,2
...,...,...
6479,7a92cb13-de00-4cf9-8f2e-fe1de7629692,3
6480,3179b05a-487a-4208-9d7a-115c9532149b,3
6481,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2
6482,9207b26d-79dc-41fc-b62c-67a6821132b9,2


In [ ]:
def group_by_user(df_attendance_house):
  df_group_by =  df_attendance_house.groupby(['user_id']).agg(

      # total_late_score = ('Late_Morning', 'Late_Evening', 'Late_Night', 'sum'),
      # total_late_early = ('Early_Morning', 'Early_Evening', 'Early_Night', 'sum'),

      total_late_morning = ('Late_Morning', 'sum'),
      total_late_night = ('Late_Night', 'sum'),
      total_late_evening = ('Late_Evening', 'sum'),

      total_late_morning_5 = ('Late_Morning_in_5', 'sum'),
      total_late_night_5 = ('Late_Nighe_in_5', 'sum'),
      total_late_evening_5 = ('Late_Evening_in_5', 'sum'),

      total_eayly_morning = ('Early_Morning', 'sum'),
      total_early_night = ('Early_Night', 'sum'),
      total_early_evening = ('Early_Evening', 'sum'),

      total_day = ('datetime_day', 'sum'),
      total_dayofweek = ('datetime_dayofweek', 'sum'),
      total_dayofyear = ('datetime_dayofyear', 'sum'),
      total_hour = ('datetime_hour', 'sum'),
      total_min = ('datetime_minute', 'sum'),

      total_intime_morning = ('In_time_Morning', 'sum'),
      total_intime_night = ('In_time_Night', 'sum'),
      total_intime_evening = ('In_time_Evening', 'sum'),
  )

  return df_group_by



df_attendance_house_agg = group_by_user(df_attendance_house)
df_attendance_house_agg

,total_late_morning,total_late_night,total_late_evening,total_late_morning_5,total_late_night_5,total_late_evening_5,total_eayly_morning,total_early_night,total_early_evening,total_day,total_dayofweek,total_dayofyear,total_hour,total_min,total_intime_morning,total_intime_night,total_intime_evening
user_id,,,,,,,,,,,,,,,,,
0492584f-5dda-4ad6-ab2f-79c8d43a8753,4,0,6,4,0,6,6,11,5,1009.0,147.0,8306.0,825.0,2784.0,15,17,17
0a3e80d5-ebfc-4fb6-9c73-eb4afcca687a,4,0,5,4,0,5,4,8,6,896.0,129.0,7406.0,734.0,2467.0,14,15,15
0e59353d-8c6d-4a18-876e-5f506a40036e,4,0,7,4,0,7,9,6,10,1000.0,144.0,8176.0,814.0,2577.0,15,17,16
0fa19346-050f-48ac-9f1b-8b214d0ee662,5,0,4,5,0,4,10,8,7,980.0,138.0,7944.0,790.0,2550.0,13,16,18
12e832b1-ba8d-48d4-acb1-67c2ed2504e4,4,0,6,4,0,6,2,9,11,986.0,138.0,8071.0,815.0,2691.0,13,16,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f7913f48-8bed-4c54-95d8-968fc64606e3,4,1,6,4,0,6,8,6,6,932.0,126.0,7382.0,750.0,2397.0,14,16,15
f82cab1a-3510-414e-a268-b461e7b9c8d3,4,0,4,4,0,4,2,3,2,727.0,80.0,6086.0,600.0,2258.0,12,13,13
f90e665a-2d98-4ab1-a6de-5ce1a5403861,4,0,6,4,0,6,7,5,8,969.0,141.0,7933.0,797.0,2677.0,13,16,17


In [ ]:
df_attendance_house_agg = df_attendance_house_agg.reset_index()
df_attendance_house_agg

,user_id,total_late_morning,total_late_night,total_late_evening,total_late_morning_5,total_late_night_5,total_late_evening_5,total_eayly_morning,total_early_night,total_early_evening,total_day,total_dayofweek,total_dayofyear,total_hour,total_min,total_intime_morning,total_intime_night,total_intime_evening
0,0492584f-5dda-4ad6-ab2f-79c8d43a8753,4,0,6,4,0,6,6,11,5,1009.0,147.0,8306.0,825.0,2784.0,15,17,17
1,0a3e80d5-ebfc-4fb6-9c73-eb4afcca687a,4,0,5,4,0,5,4,8,6,896.0,129.0,7406.0,734.0,2467.0,14,15,15
2,0e59353d-8c6d-4a18-876e-5f506a40036e,4,0,7,4,0,7,9,6,10,1000.0,144.0,8176.0,814.0,2577.0,15,17,16
3,0fa19346-050f-48ac-9f1b-8b214d0ee662,5,0,4,5,0,4,10,8,7,980.0,138.0,7944.0,790.0,2550.0,13,16,18
4,12e832b1-ba8d-48d4-acb1-67c2ed2504e4,4,0,6,4,0,6,2,9,11,986.0,138.0,8071.0,815.0,2691.0,13,16,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,f7913f48-8bed-4c54-95d8-968fc64606e3,4,1,6,4,0,6,8,6,6,932.0,126.0,7382.0,750.0,2397.0,14,16,15
98,f82cab1a-3510-414e-a268-b461e7b9c8d3,4,0,4,4,0,4,2,3,2,727.0,80.0,6086.0,600.0,2258.0,12,13,13
99,f90e665a-2d98-4ab1-a6de-5ce1a5403861,4,0,6,4,0,6,7,5,8,969.0,141.0,7933.0,797.0,2677.0,13,16,17
100,faac7a28-16cd-48ae-833d-09ea4ae49833,4,0,7,4,0,7,8,5,9,1009.0,147.0,8306.0,826.0,2546.0,14,15,16


In [ ]:
# Left Join House
house_table = df_attendance_house[["user_id", "house"]]
house_table = house_table.drop_duplicates(
    subset = ["user_id", "house"]
)

house_table

,user_id,house
0,cdd098ee-4295-4839-b1a3-4d44840feacd,2
1,2ad18754-2522-4fa9-af12-bccdf42d3905,4
2,3bd6dcac-904c-4614-be50-9c3949cb3405,4
3,1d9dd772-17c9-4c8c-a3dd-1dbfe41c5ba9,4
4,7eecd605-6ca1-4343-bf40-56ab7ea20dc9,2
...,...,...
114,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2
169,ef301f17-11a3-401c-b25b-8fea5eb1256f,4
186,332a0b3d-cc6a-4a02-a2e9-7ac0840a6069,5
588,f82cab1a-3510-414e-a268-b461e7b9c8d3,3


In [ ]:
df_attendance_house_final = pd.merge(
    df_attendance_house_agg,
    house_table,
    how='left',
    on=['user_id']
)

df_attendance_house_final

,user_id,total_late_morning,total_late_night,total_late_evening,total_late_morning_5,total_late_night_5,total_late_evening_5,total_eayly_morning,total_early_night,total_early_evening,total_day,total_dayofweek,total_dayofyear,total_hour,total_min,total_intime_morning,total_intime_night,total_intime_evening,house
0,0492584f-5dda-4ad6-ab2f-79c8d43a8753,4,0,6,4,0,6,6,11,5,1009.0,147.0,8306.0,825.0,2784.0,15,17,17,0
1,0a3e80d5-ebfc-4fb6-9c73-eb4afcca687a,4,0,5,4,0,5,4,8,6,896.0,129.0,7406.0,734.0,2467.0,14,15,15,2
2,0e59353d-8c6d-4a18-876e-5f506a40036e,4,0,7,4,0,7,9,6,10,1000.0,144.0,8176.0,814.0,2577.0,15,17,16,4
3,0fa19346-050f-48ac-9f1b-8b214d0ee662,5,0,4,5,0,4,10,8,7,980.0,138.0,7944.0,790.0,2550.0,13,16,18,1
4,12e832b1-ba8d-48d4-acb1-67c2ed2504e4,4,0,6,4,0,6,2,9,11,986.0,138.0,8071.0,815.0,2691.0,13,16,18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,f7913f48-8bed-4c54-95d8-968fc64606e3,4,1,6,4,0,6,8,6,6,932.0,126.0,7382.0,750.0,2397.0,14,16,15,3
98,f82cab1a-3510-414e-a268-b461e7b9c8d3,4,0,4,4,0,4,2,3,2,727.0,80.0,6086.0,600.0,2258.0,12,13,13,3
99,f90e665a-2d98-4ab1-a6de-5ce1a5403861,4,0,6,4,0,6,7,5,8,969.0,141.0,7933.0,797.0,2677.0,13,16,17,1
100,faac7a28-16cd-48ae-833d-09ea4ae49833,4,0,7,4,0,7,8,5,9,1009.0,147.0,8306.0,826.0,2546.0,14,15,16,2


## Prepare TestSet

In [ ]:
df_submission

,user_id,house
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,NaN
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,NaN
...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,NaN
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,NaN
65,1bedb648-5a50-4680-a09b-27240759f9a0,NaN
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,NaN


In [ ]:
df_attendance

,user_id,datetime,datetime_dt,datetime_year,datetime_month,datetime_day,datetime_hour,datetime_minute,datetime_second,datetime_dayofweek,datetime_dayofyear,datetime_quarter,datetime_is_weekend,datetime_season,datetime_is_month_start,datetime_is_month_end,datetime_is_quarter_start,datetime_is_quarter_end,datetime_hour_sin,datetime_hour_cos
0,5931fa6a-af6b-43b8-babb-28349854d406,2024-04-19 17:30:08.827555,2024-04-19 17:30:08.827555,2024.0,4.0,19.0,17.0,30.0,8.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
1,5f1a2ecf-943d-446b-b94d-d7d1ad64213d,2024-04-19 17:30:11.405020,2024-04-19 17:30:11.405020,2024.0,4.0,19.0,17.0,30.0,11.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
2,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,2024-04-19 17:30:14.433098,2024-04-19 17:30:14.433098,2024.0,4.0,19.0,17.0,30.0,14.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
3,cdd098ee-4295-4839-b1a3-4d44840feacd,2024-04-19 17:30:17.214453,2024-04-19 17:30:17.214453,2024.0,4.0,19.0,17.0,30.0,17.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
4,f922a5d4-3651-46f7-aa98-307c045dd4b5,2024-04-19 17:30:21.556805,2024-04-19 17:30:21.556805,2024.0,4.0,19.0,17.0,30.0,21.0,4.0,110.0,2.0,0,2.0,0,0,0,0,-0.965926,-0.258819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10798,ef836255-9ff1-48bd-9ee1-b3e205f96367,2024-05-17 13:57:11.458189,2024-05-17 13:57:11.458189,2024.0,5.0,17.0,13.0,57.0,11.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10799,84aea5bb-fc9f-463e-8ed9-5c36ccb0dc7f,2024-05-17 13:57:15.457449,2024-05-17 13:57:15.457449,2024.0,5.0,17.0,13.0,57.0,15.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10800,1e103917-1d29-424d-b5e1-924e0cdcd6dc,2024-05-17 13:57:19.924906,2024-05-17 13:57:19.924906,2024.0,5.0,17.0,13.0,57.0,19.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.258819,-0.965926
10801,9207b26d-79dc-41fc-b62c-67a6821132b9,2024-05-17 14:01:03.323543,2024-05-17 14:01:03.323543,2024.0,5.0,17.0,14.0,1.0,3.0,4.0,138.0,2.0,0,2.0,0,0,0,0,-0.500000,-0.866025


In [ ]:
# df_attendance_house_test = pd.merge(df_submission, df_attendance, how="inner", on=["user_id"])
df_attendance_house_test = df_submission.merge(df_attendance, how='inner', on='user_id')
df_attendance_house_test = create_attendance_bool(df_attendance_house_test)

# Drop Unused
df_attendance_house_test = df_attendance_house_test.drop(
    columns=["datetime", "datetime_dt", "datetime_month", "datetime_year", "datetime_quarter", "datetime_season", "datetime_is_month_start", "datetime_is_month_end", "datetime_is_quarter_start", "datetime_is_quarter_end"]
)

df_attendance_house_test['house'] = df_attendance_house_test['house'].map(house_mapping)
df_attendance_house_test_agg = group_by_user(df_attendance_house_test)

# Left Join House
house_test_table = df_attendance_house_test[["user_id", "house"]]
house_test_table = house_test_table.drop_duplicates(
    subset = ["user_id", "house"]
)

df_attendance_house_test_final = pd.merge(
    df_attendance_house_test_agg,
    house_test_table,
    how='left',
    on=['user_id']
)



## Train CatBoost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [ ]:
def catboost_classification_pipeline(df, target_columns, df_test):

  # Split Train ans Test
  features = df.drop(['user_id', target_columns], axis=1)
  target = df[target_columns]
  X_train, X_val, Y_train, Y_val = train_test_split(features, target, random_state=3200, test_size=0.20)

  # Defined Model
  model_catboost = CatBoostClassifier(
      iterations=500,
      learning_rate = 0.1,
      depth = 6
  )

  # Scaler data
  scaler = StandardScaler()
  scaler.fit(X_train)
  X_train = scaler.transform(X_train)
  X_val = scaler.transform(X_val)
  print(type(X_train))
  print(type(X_val))
  print(X_train.shape)
  print(X_val.shape)

  # Fit Model
  model_catboost.fit(
      X_train,
      Y_train,
      eval_set = (X_val, Y_val)
  )

  # Evaluation
  y_pred = model_catboost.predict(X_val)
  print(type(y_pred))
  print(y_pred.shape)
  print("Validation F1: ", f1_score(Y_val, y_pred, average='weighted'))

  # Preparation Test Set
  X_test = df_test.drop(['user_id', target_columns], axis=1)
  X_test = scaler.transform(X_test)
  print(f"X_test : {X_test}")

  # Prediction
  y_test_prediction = model_catboost.predict(X_test)
  print(y_test_prediction)
  print(type(y_test_prediction))

  # Format to Pandas Series
  df_pred = pd.DataFrame(y_test_prediction)
  df_test_concat = pd.concat([df_test, df_pred], axis=1)
  df_test_concat = df_test_concat.drop(
      columns = [target_columns]
  )
  print(f"df_test_concat : {df_test_concat}")

  # Rename last columns
  df_test_concat.columns = [*df_test_concat.columns[:-1], target_columns]
  df_test_concat = df_test_concat[["user_id", target_columns]]

  return df_test_concat


In [ ]:
df_attendance_house_final

,user_id,total_late_morning,total_late_night,total_late_evening,total_late_morning_5,total_late_night_5,total_late_evening_5,total_eayly_morning,total_early_night,total_early_evening,total_day,total_dayofweek,total_dayofyear,total_hour,total_min,total_intime_morning,total_intime_night,total_intime_evening,house
0,0492584f-5dda-4ad6-ab2f-79c8d43a8753,4,0,6,4,0,6,6,11,5,1009.0,147.0,8306.0,825.0,2784.0,15,17,17,0
1,0a3e80d5-ebfc-4fb6-9c73-eb4afcca687a,4,0,5,4,0,5,4,8,6,896.0,129.0,7406.0,734.0,2467.0,14,15,15,2
2,0e59353d-8c6d-4a18-876e-5f506a40036e,4,0,7,4,0,7,9,6,10,1000.0,144.0,8176.0,814.0,2577.0,15,17,16,4
3,0fa19346-050f-48ac-9f1b-8b214d0ee662,5,0,4,5,0,4,10,8,7,980.0,138.0,7944.0,790.0,2550.0,13,16,18,1
4,12e832b1-ba8d-48d4-acb1-67c2ed2504e4,4,0,6,4,0,6,2,9,11,986.0,138.0,8071.0,815.0,2691.0,13,16,18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,f7913f48-8bed-4c54-95d8-968fc64606e3,4,1,6,4,0,6,8,6,6,932.0,126.0,7382.0,750.0,2397.0,14,16,15,3
98,f82cab1a-3510-414e-a268-b461e7b9c8d3,4,0,4,4,0,4,2,3,2,727.0,80.0,6086.0,600.0,2258.0,12,13,13,3
99,f90e665a-2d98-4ab1-a6de-5ce1a5403861,4,0,6,4,0,6,7,5,8,969.0,141.0,7933.0,797.0,2677.0,13,16,17,1
100,faac7a28-16cd-48ae-833d-09ea4ae49833,4,0,7,4,0,7,8,5,9,1009.0,147.0,8306.0,826.0,2546.0,14,15,16,2


In [ ]:
df_prediction_submit = catboost_classification_pipeline(
    df_attendance_house_final, "house", df_attendance_house_test_final
)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(81, 17)
(21, 17)
0:	learn: 1.7573332	test: 1.7882212	best: 1.7882212 (0)	total: 5.84ms	remaining: 2.91s
1:	learn: 1.7197446	test: 1.7853294	best: 1.7853294 (1)	total: 9.32ms	remaining: 2.32s
2:	learn: 1.6901775	test: 1.7932270	best: 1.7853294 (1)	total: 14ms	remaining: 2.31s
3:	learn: 1.6599421	test: 1.7969562	best: 1.7853294 (1)	total: 16.8ms	remaining: 2.08s
4:	learn: 1.6316267	test: 1.7975732	best: 1.7853294 (1)	total: 18.8ms	remaining: 1.87s
5:	learn: 1.6083738	test: 1.7957759	best: 1.7853294 (1)	total: 21ms	remaining: 1.73s
6:	learn: 1.5834686	test: 1.7979220	best: 1.7853294 (1)	total: 25.5ms	remaining: 1.79s
7:	learn: 1.5601603	test: 1.7999001	best: 1.7853294 (1)	total: 31.5ms	remaining: 1.94s
8:	learn: 1.5410564	test: 1.8025258	best: 1.7853294 (1)	total: 37.1ms	remaining: 2.02s
9:	learn: 1.5169664	test: 1.8049315	best: 1.7853294 (1)	total: 41.9ms	remaining: 2.05s
10:	learn: 1.4991390	test: 1.8035939	best: 1.7853294 (1)	total: 45.4

In [ ]:
df_prediction_submit

,user_id,house
0,0cb94071-a8ec-4885-8316-e0e4fbebec32,3
1,0d9ff778-c1fe-4966-baa9-e35783893f39,5
2,12584309-4699-485d-9f5b-a57ebfc4283a,0
3,1a024edc-e397-4508-958e-7051ae278650,5
4,1bedb648-5a50-4680-a09b-27240759f9a0,3
...,...,...
63,e81fe35d-eabc-4bef-a19d-b7cf24d01366,4
64,ef836255-9ff1-48bd-9ee1-b3e205f96367,2
65,f89e8057-94eb-42d4-9ea6-c5f00419c387,2
66,f922a5d4-3651-46f7-aa98-307c045dd4b5,1


In [ ]:
key_mapping = {
  0 : 'house1',
  1 : 'house2',
  2 : 'house3',
  3 : 'house4',
  4 : 'house5',
  5 : 'house6',
}

df_prediction_submit['house'] = df_prediction_submit['house'].map(key_mapping)
df_prediction_submit

,user_id,house
0,0cb94071-a8ec-4885-8316-e0e4fbebec32,house4
1,0d9ff778-c1fe-4966-baa9-e35783893f39,house6
2,12584309-4699-485d-9f5b-a57ebfc4283a,house1
3,1a024edc-e397-4508-958e-7051ae278650,house6
4,1bedb648-5a50-4680-a09b-27240759f9a0,house4
...,...,...
63,e81fe35d-eabc-4bef-a19d-b7cf24d01366,house5
64,ef836255-9ff1-48bd-9ee1-b3e205f96367,house3
65,f89e8057-94eb-42d4-9ea6-c5f00419c387,house3
66,f922a5d4-3651-46f7-aa98-307c045dd4b5,house2


In [ ]:
df_submission

,user_id,house
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,NaN
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,NaN
...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,NaN
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,NaN
65,1bedb648-5a50-4680-a09b-27240759f9a0,NaN
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,NaN


In [ ]:
df_final_submission = pd.merge(df_submission, df_prediction_submit, on="user_id", how="inner")
df_final_submission

,user_id,house_x,house_y
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house3,house4
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,NaN,house1
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,NaN,house4
...,...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,NaN,house4
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,NaN,house1
65,1bedb648-5a50-4680-a09b-27240759f9a0,NaN,house4
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,NaN,house3


In [ ]:
df_final_submission = df_final_submission.drop(
    columns = ["house_x"]
)
# df.rename(columns={"A": "a", "B": "c"})
df_final_submission = df_final_submission.rename(
    columns = {
        "house_y" : "house"
    }
)

df_final_submission

,user_id,house
0,6e009a96-2b8b-4048-8b0c-20a7924a4b4f,house4
1,81c9f456-b5a2-4e28-ad69-eb7d89374d37,house1
2,dda733a2-2add-44a5-b888-81943923c2c6,house4
3,12584309-4699-485d-9f5b-a57ebfc4283a,house1
4,c968fef9-5411-49d5-b6e1-d9a52add4ab8,house4
...,...,...
63,fed3dced-4009-461e-9411-a440c779e32c,house4
64,60fea3c0-9066-49ea-beb2-b055ab2b4ff9,house1
65,1bedb648-5a50-4680-a09b-27240759f9a0,house4
66,ef836255-9ff1-48bd-9ee1-b3e205f96367,house3


In [ ]:
df_final_submission.to_csv("exp1_spai.csv")